In [1]:
"""
train_model.py
---------------
AI Loan Approval & Risk Prediction System
Trains a Logistic Regression model to predict loan approval.

Pipeline:
  1. Load raw data
  2. Handle missing values
  3. Encode categorical features (LabelEncoder)
  4. Scale numeric features (StandardScaler)
  5. Train/test split
  6. Train Logistic Regression
  7. Evaluate (accuracy, precision, recall, F1, confusion matrix, ROC-AUC)
  8. Save the trained model + encoders + scaler for the Streamlit app

Run:
  python train_model.py
"""

import json
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

DATA_PATH = "loan_data.csv"
MODEL_DIR = "model"
os.makedirs(MODEL_DIR, exist_ok=True)

CATEGORICAL_COLS = [
    "Gender",
    "Married",
    "Dependents",
    "Education",
    "Self_Employed",
    "Property_Area",
]
NUMERIC_COLS = [
    "ApplicantIncome",
    "CoapplicantIncome",
    "LoanAmount",
    "Loan_Amount_Term",
    "Credit_History",
]
TARGET_COL = "Loan_Status"


def load_data(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"'{path}' not found. Run `python generate_data.py` first "
            "(or drop in your own CSV with the same column names)."
        )
    return pd.read_csv(path)


def preprocess(df: pd.DataFrame):
    df = df.copy()

    # --- handle missing values ---
    for col in CATEGORICAL_COLS:
        df[col] = df[col].fillna(df[col].mode()[0])
    for col in NUMERIC_COLS:
        df[col] = df[col].fillna(df[col].median())

    # --- encode categorical columns ---
    encoders = {}
    for col in CATEGORICAL_COLS:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le

    target_encoder = LabelEncoder()
    df[TARGET_COL] = target_encoder.fit_transform(df[TARGET_COL].astype(str))  # N=0, Y=1

    # --- scale numeric columns ---
    scaler = StandardScaler()
    df[NUMERIC_COLS] = scaler.fit_transform(df[NUMERIC_COLS])

    feature_cols = CATEGORICAL_COLS + NUMERIC_COLS
    X = df[feature_cols]
    y = df[TARGET_COL]

    return X, y, encoders, scaler, target_encoder, feature_cols


def train_and_evaluate(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": round(accuracy_score(y_test, y_pred), 4),
        "precision": round(precision_score(y_test, y_pred), 4),
        "recall": round(recall_score(y_test, y_pred), 4),
        "f1_score": round(f1_score(y_test, y_pred), 4),
        "roc_auc": round(roc_auc_score(y_test, y_proba), 4),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    }

    print("\n=== Model Evaluation ===")
    for k, v in metrics.items():
        if k != "confusion_matrix":
            print(f"{k:>10}: {v}")
    print("\nConfusion Matrix (rows=actual, cols=predicted) [N, Y]:")
    print(np.array(metrics["confusion_matrix"]))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["Rejected (N)", "Approved (Y)"]))

    return model, metrics


def main():
    df = load_data(DATA_PATH)
    X, y, encoders, scaler, target_encoder, feature_cols = preprocess(df)
    model, metrics = train_and_evaluate(X, y)

    # Save everything the Streamlit app needs to reproduce this pipeline
    joblib.dump(model, os.path.join(MODEL_DIR, "loan_model.pkl"))
    joblib.dump(encoders, os.path.join(MODEL_DIR, "encoders.pkl"))
    joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.pkl"))
    joblib.dump(target_encoder, os.path.join(MODEL_DIR, "target_encoder.pkl"))
    joblib.dump(feature_cols, os.path.join(MODEL_DIR, "feature_cols.pkl"))

    with open(os.path.join(MODEL_DIR, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    print(f"\nSaved model + preprocessing artifacts to ./{MODEL_DIR}/")


if __name__ == "__main__":
    main()



=== Model Evaluation ===
  accuracy: 0.8049
 precision: 0.8427
    recall: 0.8824
  f1_score: 0.8621
   roc_auc: 0.8115

Confusion Matrix (rows=actual, cols=predicted) [N, Y]:
[[24 14]
 [10 75]]

Classification Report:
              precision    recall  f1-score   support

Rejected (N)       0.71      0.63      0.67        38
Approved (Y)       0.84      0.88      0.86        85

    accuracy                           0.80       123
   macro avg       0.77      0.76      0.76       123
weighted avg       0.80      0.80      0.80       123


Saved model + preprocessing artifacts to ./model/
